In [1]:
# connecting with GoogleDrive (~33s)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
# imports
import pandas as pd
import numpy as np
import re


In [ ]:
def df_preview(data: dict, logger=None):
    """
    Docstring for data_preview
    
    """

    for name, df in data.items():
        if logger:
            log.log_section(logger, f"EDA RAW DATA ({name})")
            logger.info("SHAPE:\t %s", df.shape) 
            logger.info("INFO:\n %s", info_as_string(df)) 
            logger.info("HEAD:\n %s %s", df.head(5), "\n") 
    
        else:
            print("\n")
            print("=" * 50 + "\n")
            print(f"--- EDA RAW DATA ({name}) --- {datetime.now().strftime("%Y-%m-%d %H:%M:%S")} ---\n")
            print("=" * 50 + "\n")
            print("SHAPE:\t", df.shape)
            print("INFO\n", info_as_string(df))
            print(f"HEAD:\n{df.head(5)}\n")

    return 

In [ ]:
# load df

path = "/home/robfra/0_Portfolio_Projekte/LLM/data/data_generic/escalation_dataset_v2.csv"

df = pd.read_csv(path)

data = {"escalation_reports": df}

df_preview(data)
# print("info:")
# df.info()
# print("\nhead:\n", df.head(5))

df_grouped = df.groupby(["domain", "clarity"])["expected_action"].value_counts()
print("\n")
print("=" * 50 + "\n")
print("grouped by 'domain / clarity':")
print("=" * 50 + "\n")
print(df_grouped)

info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   case_id          320 non-null    int64 
 1   domain           320 non-null    object
 2   clarity          320 non-null    object
 3   expected_action  320 non-null    object
 4   report_text      320 non-null    object
 5   comment          320 non-null    object
dtypes: int64(1), object(5)
memory usage: 15.1+ KB

head:
    case_id   domain    clarity expected_action  \
0        1    psych      clear      escalation   
1        2    psych      clear   no_escalation   
2        3    psych  ambiguous      escalation   
3        4    psych  ambiguous   no_escalation   
4        5  somatic      clear      escalation   

                                         report_text  \
0  The patient reports escalating psychological d...   
1  The patient describes emotional strain related...   
2 

# Output:
```



# (1) Grundlegende Textstruktur

In [ ]:
# (A) Minimal-Setup & “sanity checks”

def basic_text_structure(df, text_col, label_col):
    # (1) base checks
    assert text_col in df.columns, f"Missing text col: {text_col}"
    assert label_col in df.columns, f"Missing label col: {label_col}"

    # (2) copy + standardise
    d = df.copy()
    d[text_col] = d[text_col].astype("string")

    # 
    n_label = d[label_col].value_counts(dropna=False)
    mean_nan = d[text_col].isna().mean()
    mean_empty_text = (d[text_col].str.len() == 0).mean()

    print("unique labels + counts:\n", n_label)
    print("\nmean_nan_values:", mean_nan)
    print("\nmean_empty_strings:", mean_empty_text)

    return 

text_col = "report_text"
label_col = "expected_action"


unique labels + counts:
 expected_action
escalation       160
no_escalation    160
Name: count, dtype: int64

mean_nan_values: 0.0

mean_empty_strings: 0.0


# Output:
```



In [ ]:
# (B-1) Text-Metriken berechnen (Zeichen, Wörter, Sätze, Sonderzeichen)

def compile_text_metrics(df_in, verbose=False, group_col=None):
      # Helper
      _word_re = re.compile(r"\b\w+\b", flags=re.UNICODE)
      _sentence_re = re.compile(r"[.!?]+")
      _digit_re = re.compile(r"\d")
      _upper_re = re.compile(r"[A-ZÄÖÜ]")
      _punct_re = re.compile(r"[^\w\s]", flags=re.UNICODE)

      df = df_in.copy()
      # Basic length metrics
      df["n_chars"] = df[TEXT_COL].str.len().fillna(0).astype(int)
      df["n_words"] = df[TEXT_COL].apply(lambda x: len(_word_re.findall(x)) if x else 0)
      df["n_sentences"] = df[TEXT_COL].apply(lambda x: len(_sentence_re.findall(x)) if x else 0)

      # Ratios / counts
      df["n_digits"] = df[TEXT_COL].apply(lambda x: len(_digit_re.findall(x)) if x else 0)
      df["n_upper"]  = df[TEXT_COL].apply(lambda x: len(_upper_re.findall(x)) if x else 0)
      df["n_punct"]  = df[TEXT_COL].apply(lambda x: len(_punct_re.findall(x)) if x else 0)

      df["digit_ratio"] = np.where(df["n_chars"] > 0, 
                                    df["n_digits"] / df["n_chars"], 
                                    0.0)
      df["upper_ratio"] = np.where(df["n_chars"] > 0, 
                                    df["n_upper"]  / df["n_chars"], 
                                    0.0)
      df["punct_ratio"] = np.where(df["n_chars"] > 0, 
                                    df["n_punct"]  / df["n_chars"], 
                                    0.0)

      # 
      if verbose:
            metrics = ["n_chars", "n_words",
                  "n_sentences", "n_digits",
                  "n_upper", "n_punct",
                  "digit_ratio","upper_ratio",
                  "punct_ratio"]
     
            print("text metrics:\n(1) overall:\n",d_metrics.describe().round(3))
            print(f"\n{'---'*50}")

            if group_col:
                  print("\n(2) per group:")
                  col_no = 1
                  for col in metrics:
                        print(f"(2_{col_no}) {col}:\n",
                              d_metrics.groupby(group_col)[col].describe().round(3))
                        print()

                  col_no += 1
      
      return df


group_cols=["domain", "clarity"]
df = compile_text_metrics(df_in, verbose=True, group_col=group_cols):


# cols=["domain", "clarity",
#       "n_chars", "n_words",
#       "n_sentences", "n_digits",
#       "n_upper", "n_punct",
#       "digit_ratio", "upper_ratio",
#       "punct_ratio"]


# d_metrics = d[cols]


# 	tokens
# count	320.000
# mean	27.625
# std	3.503
# min	24.000
# 25%	24.000
# 50%	27.500
# 75%	29.500
# max	34.000


text metrics:
(1) overall:
        n_chars  n_words  n_sentences  n_digits  n_upper  n_punct  digit_ratio  \
count  320.000  320.000      320.000   320.000  320.000  320.000      320.000   
mean   214.375   28.750        1.750     1.750    1.750    4.875        0.008   
std     26.284    3.387        0.434     0.434    0.434    1.055        0.002   
min    182.000   25.000        1.000     1.000    1.000    4.000        0.004   
25%    190.750   25.750        1.750     1.750    1.750    4.000        0.007   
50%    210.500   28.500        2.000     2.000    2.000    4.500        0.009   
75%    228.250   30.500        2.000     2.000    2.000    5.250        0.010   
max    256.000   35.000        2.000     2.000    2.000    7.000        0.011   

       upper_ratio  punct_ratio  
count      320.000      320.000  
mean         0.008        0.023  
std          0.002        0.003  
min          0.005        0.018  
25%          0.007        0.020  
50%          0.008        0.022  
75% 

# Output:
```



In [63]:
# (B-2)

metrics = ["n_chars", "n_words",
           "n_sentences", "n_digits",
           "n_upper", "n_punct",
           "digit_ratio","upper_ratio",
           "punct_ratio"]

print("Overall:\n", d[metrics].describe(percentiles=[0.01,0.05,0.5,0.95,0.99]).round(3).T)
print(f"\n{'---'*50}")
# print("")
print("per label:\n")
for col in metrics:
  print(f"{col}:\n", d.groupby("expected_action")[col].describe(percentiles=[0.01,0.05,0.5,0.95,0.99]).round(3))
  print(f"\n{'---'*50}\n")

print("per group:")
for col in metrics:
  print(f"{col}:\n", d.groupby(["domain", "clarity"])[col].describe(percentiles=[0.01,0.05,0.5,0.95,0.99]).round(3))
  print(f"\n{'---'*50}\n")

# summary


Overall:
              count     mean     std      min       1%       5%      50%  \
n_chars      320.0  214.375  26.284  182.000  182.000  183.000  210.500   
n_words      320.0   28.750   3.387   25.000   25.000   25.000   28.500   
n_sentences  320.0    1.750   0.434    1.000    1.000    1.000    2.000   
n_digits     320.0    1.750   0.434    1.000    1.000    1.000    2.000   
n_upper      320.0    1.750   0.434    1.000    1.000    1.000    2.000   
n_punct      320.0    4.875   1.055    4.000    4.000    4.000    4.500   
digit_ratio  320.0    0.008   0.002    0.004    0.004    0.004    0.009   
upper_ratio  320.0    0.008   0.002    0.005    0.005    0.005    0.008   
punct_ratio  320.0    0.023   0.003    0.018    0.018    0.018    0.022   

                 95%      99%      max  
n_chars      256.000  256.000  256.000  
n_words       35.000   35.000   35.000  
n_sentences    2.000    2.000    2.000  
n_digits       2.000    2.000    2.000  
n_upper        2.000    2.000    2

The patient reports persistent emotional exhaustion and social withdrawal, noting that daily demands feel increasingly unmanageable. No explicit self-harm statements were made, but concern was expressed about coping capacity if symptoms worsen (case 0).

The patient reports persistent emotional exhaustion and social withdrawal, noting that daily demands feel increasingly unmanageable. No explicit self-harm statements were made, but concern was expressed about coping capacity if symptoms worsen (case 1).

The patient reports persistent emotional exhaustion and social withdrawal, noting that daily demands feel increasingly unmanageable. No explicit self-harm statements were made, but concern was expressed about coping capacity if symptoms worsen (case 2).

In [42]:
# (C) “Extremfälle” finden (die dir später alles kaputt machen)

# Sehr kurze / sehr lange Texte
d["is_empty"] = d["n_chars"] == 0
d["is_tiny"] = d["n_words"] <= 3
d["is_huge"] = d["n_words"] >= d["n_words"].quantile(0.99)

print("Extreme reports:\n", d[["is_empty","is_tiny","is_huge"]].mean())

if len(d["is_empty"]) > 0:
  print("\n'empty' reports:\n", d.loc[d["is_empty"], [LABEL_COL, TEXT_COL]].head(10))

if len(d["is_tiny"]) > 0:
  print("\n'tiny' reports:\n", d.loc[d["is_tiny"], [LABEL_COL, TEXT_COL]].head(10))

if len(d["is_huge"]) > 0:
  print("\n'huge' reports:\n", d.loc[d["is_huge"], ["domain", "clarity", LABEL_COL, TEXT_COL]].head(3))



Extreme reports:
 is_empty    0.000
is_tiny     0.000
is_huge     0.125
dtype: float64

'empty' reports:
 Empty DataFrame
Columns: [expected_action, report_text]
Index: []

'tiny' reports:
 Empty DataFrame
Columns: [expected_action, report_text]
Index: []

'huge' reports:
                                           report_text
2   The patient reports persistent emotional exhau...
10  The patient reports persistent emotional exhau...
18  The patient reports persistent emotional exhau...


In [44]:
# (D) Verteilung

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression

X = d[["n_words","n_chars","n_sentences","digit_ratio","upper_ratio","punct_ratio"]].to_numpy()
y = d[LABEL_COL].to_numpy()

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = []
for tr, te in cv.split(X, y):
    clf = LogisticRegression(max_iter=2000)
    clf.fit(X[tr], y[tr])
    pred = clf.predict(X[te])
    # Fix: Specify pos_label for binary classification with string labels
    scores.append(f1_score(y[te], pred, average="binary" if len(np.unique(y))==2 else "macro", pos_label='escalation'))

np.mean(scores), np.std(scores)


(np.float64(0.5735802355657077), np.float64(0.11820965002479308))

In [45]:
d["text_hash"] = d["report_text"].str.lower().str.strip().apply(hash)
d["text_hash"].duplicated().mean()


np.float64(0.0)


per group:
 domain                psych            somatic         
clarity           ambiguous    clear ambiguous    clear
n_chars     count    80.000   80.000    80.000   80.000
            mean    221.750  237.750   187.250  210.750
            std      32.205   18.119     4.549    6.054
            min     189.000  219.000   182.000  204.000
            25%     190.000  220.000   183.000  205.000
...                     ...      ...       ...      ...
punct_ratio min       0.021    0.018     0.021    0.018
            25%       0.021    0.018     0.021    0.018
            50%       0.024    0.021     0.024    0.021
            75%       0.028    0.023     0.027    0.024
            max       0.028    0.024     0.027    0.025

[72 rows x 4 columns]


# (2) Tokenisierung (minimal, reproduzierbar)

,tokens
count,320.000
mean,27.625
std,3.503
min,24.000
25%,24.000
50%,27.500
75%,29.500
max,34.000


In [82]:
import re
from collections import Counter

TOKEN_RE = re.compile(r"\b[a-zA-Z]{2,}\b")

def tokenize(text):
    return TOKEN_RE.findall(text.lower())

d["tokens"] = d["report_text"].apply(tokenize)

all = d["tokens"].str.len().describe().round(3)
print("overall:\n", all)
print(f"\n{'---'*50}\n")

token_counts = {}

print("per group")
for label, group in d.groupby(["domain", "clarity"]):
  print(f"{label[0]} - {label[1]}")
  stat_token = d["tokens"].str.len().describe().round(3)
  print(stat_token)
  print()
  tokens = [t for row in group["tokens"] for t in row]
  n_tokens = Counter(tokens)
  token_counts[label] = n_tokens

  for tok in n_tokens.most_common(20):
    print(tok)
  print(f"\n{'---'*50}\n")

print(f"\n{'---'*50}\n")
print("per label")
for label, group in d.groupby("expected_action"):
  print(label)
  tokens = [t for row in group["tokens"] for t in row]
  n_tokens = Counter(tokens)
  token_counts[label] = n_tokens

  for tok in n_tokens.most_common(20):
    print(tok)
  print(f"\n{'---'*50}\n")

# for label in token_counts.keys():
#   token = token_counts[label].most_common(20)
#   print(f"token in '{label}'\n", {})
  # top_no = token_counts["no_escalation"].most_common(20)

# top_escalation, top_no
# somatic - clear, somatic - ambiguous, psych - clear, psych - ambiguous

overall:
 count    320.000
mean      27.625
std        3.503
min       24.000
25%       24.000
50%       27.500
75%       29.500
max       34.000
Name: tokens, dtype: float64

------------------------------------------------------------------------------------------------------------------------------------------------------

per group
psych - ambiguous
count    320.000
mean      27.625
std        3.503
min       24.000
25%       24.000
50%       27.500
75%       29.500
max       34.000
Name: tokens, dtype: float64

('and', 120)
('the', 80)
('patient', 80)
('reports', 80)
('emotional', 80)
('case', 80)
('persistent', 40)
('exhaustion', 40)
('social', 40)
('withdrawal', 40)
('noting', 40)
('that', 40)
('daily', 40)
('demands', 40)
('feel', 40)
('increasingly', 40)
('unmanageable', 40)
('no', 40)
('explicit', 40)
('self', 40)

------------------------------------------------------------------------------------------------------------------------------------------------------

psych - cle

In [96]:
import math

def log_odds(token, c1, c2, alpha=1):
    return math.log(
        (c1[token] + alpha) / (sum(c1.values()) + alpha)
        /
        ((c2[token] + alpha) / (sum(c2.values()) + alpha))
    )

tokens_labels = []
tokens_soma = []
tokens_psych = []
for key, value in token_counts.items():
  if key == "escalation":
    tokens_labels.append(value)
  elif key == "no_escalation":
    tokens_labels.append(value)
  elif key == ("somatic", "clear"):
    tokens_soma.append(value)
  elif key == ("somatic", "ambiguous"):
    tokens_soma.append(value)
  elif key == ("psych", "clear"):
    tokens_psych.append(value)
  elif key == ("psych", "ambiguous"):
    tokens_psych.append(value)

for names, toks_list_of_counters in zip([("escalation", "no_escalation"),
                        (('somatic', 'clear'), ('somatic', 'ambiguous')),
                         (('psych', 'clear'), ('psych', 'ambiguous'))],
                          [tokens_labels,
                           tokens_soma,
                           tokens_psych]):
  # Extract unique string tokens from the two Counter objects
  all_unique_tokens = set(toks_list_of_counters[0].keys()) | set(toks_list_of_counters[1].keys())

  log_odd_scores = [
    (t, log_odds(t, token_counts[names[0]], token_counts[names[1]]))
    for t in all_unique_tokens # Iterate over the actual string tokens
                  ]

  sort_marker = sorted(log_odd_scores, key=lambda x: x[1], reverse=True)
  print(f"relative ({names[0]} vs. {names[1]})")
  print("top 15:")
  for mark in sort_marker[:15]:
    print(f"{mark[0]}\t--> {mark[1]:.3f}")

  print("\nbottom 15:")
  for mark in sort_marker[-15:]: # Fixed the slicing for bottom 15
    print(f"{mark[0]}\t--> {mark[1]:.3f}")

  print(f"\n{'---'*50}\n")


relative (escalation vs. no_escalation)
top 15:
including	--> 4.259
over	--> 4.259
harm	--> 4.259
self	--> 4.259
worsening	--> 4.259
diagnostic	--> 3.578
about	--> 3.578
serious	--> 3.578
gradual	--> 3.578
distress	--> 3.578
regulation	--> 3.578
safety	--> 3.578
exhaustion	--> 3.578
causes	--> 3.578
was	--> 3.578

bottom 15:
factors	--> -3.849
have	--> -3.849
nonspecific	--> -3.849
management	--> -3.849
consistent	--> -3.849
work	--> -3.849
periods	--> -3.849
fatigue	--> -4.530
remains	--> -4.530
warning	--> -4.530
signs	--> -4.530
or	--> -4.530
physical	--> -4.530
functional	--> -4.932
are	--> -4.932

------------------------------------------------------------------------------------------------------------------------------------------------------

relative (('somatic', 'clear') vs. ('somatic', 'ambiguous'))
top 15:
diagnostic	--> 3.578
report	--> 3.578
serious	--> 3.578
including	--> 3.578
chronic	--> 3.578
new	--> 3.578
effective	--> 3.578
breath	--> 3.578
causes	--> 3.578
diagnos

In [98]:
#

NEGATIONS = {"no", "not", "never", "without", "unable"}
MODALS = {"must", "should", "cannot", "could", "may", "might", "requires"}

def count_set(tokens, vocab):
    return sum(t in vocab for t in tokens)

d["n_negations"] = d["tokens"].apply(lambda x: count_set(x, NEGATIONS))
d["n_modals"] = d["tokens"].apply(lambda x: count_set(x, MODALS))

d.groupby("expected_action")[["n_negations","n_modals"]].describe()


n_negations                                        n_modals  \
                      count mean      std  min  25%  50%  75%  max    count   
expected_action                                                               
escalation            160.0  0.5  0.50157  0.0  0.0  0.5  1.0  1.0    160.0   
no_escalation         160.0  0.5  0.50157  0.0  0.0  0.5  1.0  1.0    160.0   

                                                    
                mean  std  min  25%  50%  75%  max  
expected_action                                     
escalation       0.0  0.0  0.0  0.0  0.0  0.0  0.0  
no_escalation    0.0  0.0  0.0  0.0  0.0  0.0  0.0

In [ ]:
3️⃣ Label-Leakage & Shortcut-Features (EXTREM wichtig)

4️⃣ Semantische Kohärenz (LLM-relevant!)

Sprachliche Marker für „Eskalation“ (inhaltlich!)